In [1]:
import nltk
from nltk.corpus import treebank
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# Download the Treebank corpus and Universal tagset if you haven't already
nltk.download('treebank')
nltk.download('universal_tagset')

[nltk_data] Downloading package treebank to /home/tamim/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/tamim/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

Extract featres and convert the data into required format for training

In [2]:
# Load the Treebank corpus with the 'universal' tagset, which simplifies the POS tags into a universal set
tagged_sentences = treebank.tagged_sents(tagset='universal')

# Function to extract features for each word in a sentence
# Features can include the word itself, prefix, suffix, capitalization, position in the sentence, etc.
def extract_features(sentence, i):
    word = sentence[i][0] # Extract the word at position i
    # Create a dictionary of features for the word
    features = {
        'word': word,                                # The word itself
        'is_first': i == 0,                          # Boolean: True if this is the first word in the sentence
        'is_last': i == len(sentence) - 1,           # Boolean: True if this is the last word in the sentence
        'is_capitalized': word[0].upper() == word[0],# Boolean: True if the first letter is capitalized
        'is_all_caps': word.upper() == word,         # Boolean: True if all letters are capitalized
        'is_all_lower': word.lower() == word,        # Boolean: True if all letters are lowercase
        'prefix-1': word[0],                         # The first character of the word (1-character prefix)
        'prefix-2': word[:2],                        # The first two characters of the word (2-character prefix)
        'prefix-3': word[:3],                        # The first three characters of the word (3-character prefix)
        'suffix-1': word[-1],                        # The last character of the word (1-character suffix)
        'suffix-2': word[-2:],                       # The last two characters of the word (2-character suffix)
        'suffix-3': word[-3:],                       # The last three characters of the word (3-character suffix)
        'prev_word': '' if i == 0 else sentence[i - 1][0], # The previous word (empty if this is the first word)
        'next_word': '' if i == len(sentence) - 1 else sentence[i + 1][0], # The next word (empty if this is the last word)
        'is_numeric': word.isdigit(),                # Boolean: True if the word consists of digits (a number)
        'capitals_inside': word[1:].lower() != word[1:] # Boolean: True if there are capital letters inside the word
    }
    return features

# Function to transform a corpus of tagged sentences into features (X) and labels (y)
# Each sentence is processed word by word to extract features and associated POS tags
def transform_to_dataset(tagged_sentences):
    X, y = [], []  # Initialize empty lists for features and labels

    # Loop over each sentence in the corpus
    for sentence in tagged_sentences:
        # Loop over each word in the sentence
        for i in range(len(sentence)):
            X.append(extract_features(sentence, i))  # Extract features for the current word and append to X
            y.append(sentence[i][1])                 # Append the POS tag (label) for the word to y

    return X, y  # Return the feature set and labels

# Transform the entire set of tagged sentences into a feature set (X) and labels (y)
X, y = transform_to_dataset(tagged_sentences)

# Print the size of the feature set (number of words in total)
print(len(X))
# Print the size of the label set (should match the size of X)
print(len(y))

## Taking the first 10,000 entries from X and y for demonstration purposes
X = X[:10000]
y = y[:10000]

# Print the size of the truncated feature set
print(len(X))
# Print the size of the truncated label set
print(len(y))
# print the first feature set (features for the first word)
print(X[0])

100676
100676
10000
10000
{'word': 'Pierre', 'is_first': True, 'is_last': False, 'is_capitalized': True, 'is_all_caps': False, 'is_all_lower': False, 'prefix-1': 'P', 'prefix-2': 'Pi', 'prefix-3': 'Pie', 'suffix-1': 'e', 'suffix-2': 're', 'suffix-3': 'rre', 'prev_word': '', 'next_word': 'Vinken', 'is_numeric': False, 'capitals_inside': False}


Convert into vectors and split the dataset into train & test

In [3]:
# Convert features to a format usable by the model
vec = DictVectorizer(sparse=True)
X = vec.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Train and test the model using Linear Regration technique

In [4]:
# Initialize and train the Logistic Regression model
logreg = LogisticRegression(max_iter=100)
logreg.fit(X_train, y_train)

# Predict on the test set
y_pred = logreg.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9345
Classification Report:
               precision    recall  f1-score   support

           .       1.00      1.00      1.00       219
         ADJ       0.87      0.67      0.76       137
         ADP       0.97      0.96      0.96       210
         ADV       0.91      0.72      0.81        58
        CONJ       1.00      0.98      0.99        44
         DET       0.98      0.99      0.98       161
        NOUN       0.89      0.96      0.92       604
         NUM       0.99      0.99      0.99        89
        PRON       0.97      1.00      0.99        39
         PRT       0.93      0.96      0.94        52
        VERB       0.91      0.89      0.90       253
           X       1.00      1.00      1.00       134

    accuracy                           0.93      2000
   macro avg       0.95      0.93      0.94      2000
weighted avg       0.93      0.93      0.93      2000



Train and test the model using K-Nearest Neighbors (KNN)

In [5]:
# import KNN classifier
from sklearn.neighbors import KNeighborsClassifier

# Initialize and train the K-Nearest Neighbors model
knn = KNeighborsClassifier(n_neighbors=5) # You can adjust k (n_neighbors) as needed
knn.fit(X_train, y_train)

# Predict on the test set
y_pred = knn.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.897
Classification Report:
               precision    recall  f1-score   support

           .       1.00      1.00      1.00       219
         ADJ       0.70      0.61      0.65       137
         ADP       0.88      0.95      0.92       210
         ADV       0.82      0.62      0.71        58
        CONJ       0.93      0.98      0.96        44
         DET       0.95      0.94      0.95       161
        NOUN       0.87      0.92      0.89       604
         NUM       0.99      0.96      0.97        89
        PRON       0.95      0.92      0.94        39
         PRT       0.94      0.92      0.93        52
        VERB       0.86      0.79      0.82       253
           X       1.00      1.00      1.00       134

    accuracy                           0.90      2000
   macro avg       0.91      0.88      0.89      2000
weighted avg       0.90      0.90      0.89      2000

